Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import random

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    IA3Config,
    PrefixTuningConfig,
    get_peft_model
)
import evaluate
from transformers import Trainer
import torch
from torch.nn import CrossEntropyLoss

class CoLATrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,  # 🔥 ADD THIS
    ):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = torch.tensor([1.0, 2.0], device=logits.device)
        loss_fct = CrossEntropyLoss(weight=weights)

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss



In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


Model

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding=True,
        max_length=128
    )


Dataset (MRPC)

In [ ]:
task = "cola"
dataset = load_dataset("glue", task)
metric = evaluate.load("glue", task)
num_labels = 2


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

Dataset helper

In [ ]:
def get_subset(dataset, fraction, seed=42):
    size = int(len(dataset) * fraction)
    return dataset.shuffle(seed=seed).select(range(size))

def build_datasets(fraction):
    small_train = get_subset(dataset["train"], fraction)

    train_ds = small_train.map(preprocess, batched=True)
    val_ds = dataset["validation"].map(preprocess, batched=True)

    keep_cols = ["input_ids", "attention_mask", "label"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

    train_ds = train_ds.rename_column("label", "labels")
    val_ds = val_ds.rename_column("label", "labels")

    train_ds.set_format("torch")
    val_ds.set_format("torch")

    return train_ds, val_ds

Metrices

In [ ]:
from sklearn.metrics import matthews_corrcoef

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    print("Pred distribution:", np.bincount(preds))
    print("Gold distribution:", np.bincount(labels))

    return {
        "matthews_correlation": matthews_corrcoef(labels, preds)
    }



Trainable param

In [ ]:
def print_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

**Training arguments**

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

training_args = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=2e-4,      # ↑ higher than full FT
    warmup_ratio=0.1,        # 🔑 prevents early collapse
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.0,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)


Train

In [ ]:
def train_and_eval(model, name, train_ds, val_ds):
    print(f"\n===== Training: {name} =====")
    print_trainable_params(model)

    if name == "Full FT":
        args = training_args_full
    elif name == "BitFit":
        args = training_args          # 🔑 this must hit here
    else:
        args = training_args          # LoRA / IA3 / Prefix

    trainer = CoLATrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    return trainer.evaluate()


Model

In [ ]:
from peft import TaskType

def build_models():
    # Full FT
    model_full = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )

    # BitFit
    model_bitfit = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )
    for name, param in model_bitfit.named_parameters():
        if "bias" not in name:
            param.requires_grad = False

    # LoRA
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["query", "value"],
        lora_dropout=0.1,
        task_type=TaskType.SEQ_CLS,
    )
    model_lora = get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels),
        lora_config
    )

    # IA3
    ia3_config = IA3Config(task_type=TaskType.SEQ_CLS)
    model_ia3 = get_peft_model(
        AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels),
        ia3_config
    )

    # Prefix (FIXED, base frozen)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_labels
    )
    for param in base_model.parameters():
        param.requires_grad = False

    prefix_config = PrefixTuningConfig(
        task_type=TaskType.SEQ_CLS,
        num_virtual_tokens=30,
        prefix_projection=True,
        encoder_hidden_size=768,
    )
    model_prefix = get_peft_model(base_model, prefix_config)

    return model_full, model_bitfit, model_lora, model_ia3, model_prefix


Dataset fraction

In [ ]:
training_args_peft = training_args

fractions = [0.7, 1.0]
all_results = {}

for frac in fractions:
    print(f"\n===== Training with {int(frac*100)}% data =====")

    train_ds, val_ds = build_datasets(frac)
    model_full, model_bitfit, model_lora, model_ia3, model_prefix = build_models()

    results_frac = {}
    results_frac["Full FT"] = train_and_eval(model_full, "Full FT", train_ds, val_ds)
    results_frac["BitFit"]  = train_and_eval(model_bitfit, "BitFit", train_ds, val_ds)
    results_frac["LoRA"]    = train_and_eval(model_lora, "LoRA", train_ds, val_ds)
    results_frac["IA3"]     = train_and_eval(model_ia3, "IA3", train_ds, val_ds)
    results_frac["Prefix"] = train_and_eval(model_prefix, "Prefix", train_ds, val_ds)

    all_results[int(frac*100)] = results_frac

    del model_full, model_bitfit, model_lora, model_ia3, model_prefix
    torch.cuda.empty_cache()



===== Training with 70% data =====


Map:   0%|          | 0/5985 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Pred distribution: [   0 1043]
Gold distribution: [322 721]
Pred distribution: [   0 1043]
Gold distribution: [322 721]
Pred distribution: [   0 1043]
Gold distribution: [322 721]
Pred distribution: [  20 1023]
Gold distribution: [322 721]
Pred distribution: [  39 1004]
Gold distribution: [322 721]
Pred distribution: [ 65 978]
Gold distribution: [322 721]
Pred distribution: [ 65 978]
Gold distribution: [322 721]
Pred distribution: [ 75 968]
Gold distribution: [322 721]
Pred distribution: [ 66 977]
Gold distribution: [322 721]
Pred distribution: [ 86 957]
Gold distribution: [322 721]
Pred distribution: [ 88 955]
Gold distribution: [322 721]
Pred distribution: [ 79 964]
Gold distribution: [322 721]
Pred distribution: [ 75 968]
Gold distribution: [322 721]
Pred distribution: [ 85 958]
Gold distribution: [322 721]
Pred distribution: [ 82 961]
Gold distribution: [322 721]


Pred distribution: [ 82 961]
Gold distribution: [322 721]

===== Training: Prefix =====
Trainable params: 14,789,378 / 124,273,156 (11.9007%)


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.434792,0.460156,0.316347
2,0.383394,0.355597,0.447469
3,0.354239,0.342687,0.474880
4,0.302774,0.386446,0.431062
5,0.327505,0.359275,0.525898
6,0.290069,0.361461,0.496046
7,0.268625,0.379033,0.501455
8,0.217446,0.391487,0.515220
9,0.204309,0.400510,0.514531
10,0.178732,0.392613,0.523620


Pred distribution: [ 50 993]
Gold distribution: [322 721]
Pred distribution: [152 891]
Gold distribution: [322 721]
Pred distribution: [147 896]
Gold distribution: [322 721]
Pred distribution: [114 929]
Gold distribution: [322 721]
Pred distribution: [201 842]
Gold distribution: [322 721]
Pred distribution: [159 884]
Gold distribution: [322 721]
Pred distribution: [161 882]
Gold distribution: [322 721]
Pred distribution: [164 879]
Gold distribution: [322 721]
Pred distribution: [168 875]
Gold distribution: [322 721]
Pred distribution: [186 857]
Gold distribution: [322 721]
Pred distribution: [204 839]
Gold distribution: [322 721]
Pred distribution: [175 868]
Gold distribution: [322 721]
Pred distribution: [203 840]
Gold distribution: [322 721]
Pred distribution: [210 833]
Gold distribution: [322 721]
Pred distribution: [208 835]
Gold distribution: [322 721]


Pred distribution: [208 835]
Gold distribution: [322 721]
